# 05 · Validation

Milestone 6. Can the results be trusted? This notebook checks the whole system end to end:

| Check | Section |
|---|---|
| 6.1 Reproducible from a clean clone | 1 |
| 6.2 No leakage: ablation and pipeline placement | 2 |
| 6.3 Every strategy respects every constraint | 3 |
| 6.4 Backtest on the actual outcomes | 4 |
| 6.5 Robustness: GA seeds and economic assumptions | 5 |
| 6.6 Fairness monitor | 6 |

*All economic figures are illustrative assumptions, not bank performance.*

In [1]:
from dataclasses import replace
import json, subprocess, sys
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from pacopt import config as cfg, evaluation as ev, features as ft, model as m, optimize as op, pipeline as pl, charts as ch

pio.renderers.default = 'notebook_connected'
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 25)
t = pl.load_campaign_table()
comp = op.with_optimisers(t, op.compare_strategies(t, run_genetic=False, run_milp=False), *pl.load_optimisation(t))
train, test, universe = m.load_processed()
print(t.shape, '| GA and MILP results loaded from artifacts (fingerprint match)')

(10000, 53) | GA and MILP results loaded from artifacts (fingerprint match)


## 1. Reproducible from a clean clone (6.1)
`scripts/check_reproducibility.py` copies only the files git would commit (no processed data, no artifacts) into a fresh folder, runs `python -m pacopt.pipeline` there from the raw CSV, and compares every headline output with this checkout. It reuses the project environment instead of a second `uv sync`, and `uv lock --check` confirms that the lockfile matches `pyproject.toml`.

In [2]:
r = subprocess.run([sys.executable, str(cfg.PROJECT_ROOT / 'scripts' / 'check_reproducibility.py')],
                   capture_output=True, text=True, cwd=cfg.PROJECT_ROOT)
out = r.stdout
print(out[out.find('Pipeline from raw CSV'):])

Pipeline from raw CSV in the clone: 70s
                             check    this checkout      clean clone  match
               metrics: LR roc_auc         0.752621         0.752621   True
               metrics: RF roc_auc         0.755752         0.755752   True
                metrics: LR pr_auc         0.380208         0.380208   True
                metrics: RF pr_auc           0.3962           0.3962   True
                 metrics: LR brier         0.087482         0.087482   True
                 metrics: RF brier         0.086545         0.086545   True
metrics: LR max_calibration_gap_pp          1.22048          1.22048   True
metrics: RF max_calibration_gap_pp          1.98436          1.98436   True
              campaign table shape      (10000, 53)      (10000, 53)   True
  campaign table EP (max abs diff)              0.0              0.0   True
                segments identical                                     True
                  strategy: Random         60659

## 2. Leakage checks (6.2)
### 2.1 Ablation
The same tuned LR, fitted three times. Only the input columns change. `duration` is the length of the call, known only after the call has happened. `campaign`, `day` and `month` describe *this* campaign's calls (D-004).

In [3]:
ab = ev.leakage_ablation(train, test)
ab[['leaky_columns_used', 'n_features', 'roc_auc', 'pr_auc', 'precision_at_1000', 'roc_auc_vs_chosen', 'pr_auc_vs_chosen']].round(4)

,leaky_columns_used,n_features,roc_auc,pr_auc,precision_at_1000,roc_auc_vs_chosen,pr_auc_vs_chosen
variant,,,,,,,
"Chosen model: duration, campaign, day, month removed",none,15,0.7528,0.3817,0.494,0.0000,0.0000
"Only duration removed (campaign, day, month added)","campaign, day, month",18,0.7764,0.4224,0.547,0.0236,0.0406
"Everything included (duration, campaign, day, month)","campaign, day, duration, month",19,0.9080,0.5488,0.636,0.1552,0.1670


In [4]:
fig = go.Figure()
labels = ['Chosen model', '+ campaign, day, month', '+ duration too']
for metric, color, name in (('roc_auc', ch.BLUE, 'ROC-AUC'), ('pr_auc', ch.ORANGE, 'PR-AUC')):
    fig.add_bar(x=labels, y=ab[metric], name=name, marker=dict(color=color, cornerradius=4),
                text=[f'{v:.3f}' for v in ab[metric]], textposition='outside', textfont=dict(color=ch.INK_2),
                hovertemplate='%{x}<br>' + name + ' %{y:.3f}<extra></extra>')
ch.style(fig, 'Test-set performance with and without post-campaign columns', height=400)
fig.update_layout(barmode='group', bargap=0.3, bargroupgap=0.08)
fig.update_yaxes(range=[0, 1.05], title='Score')
fig.show()

Adding `duration` lifts ROC-AUC from about 0.75 to about 0.91. That is the textbook leak in this dataset: a model built that way would look excellent and fail on the morning of a real campaign, because nobody knows the call length before calling. The excluded campaign columns add a smaller but real +0.02. Our model gives up both on purpose.

### 2.2 Every fitted transform sits inside the sklearn Pipeline, after the split

In [5]:
b = m.load_model()
pd.DataFrame({k: ev.transforms_inside_pipeline(b[k]) for k in ('lr', 'rf')}).T

,n_fitted_pipelines,steps,preprocessing_inside_pipeline
lr,5,"[prep, clf]",True
rf,5,"[prep, clf]",True


In [6]:
print('Model features = allow-list:', b['features'] == ft.MODEL_FEATURES)
print('Forbidden columns in features:', sorted(set(b['features']) & cfg.FORBIDDEN_FEATURES) or 'none')
print('Universe customers in training data:', len(set(universe.customer_id) & set(train.customer_id)))

Model features = allow-list: True
Forbidden columns in features: none
Universe customers in training data: 0


The scaler and one-hot encoder are the `prep` step of each calibrated fold's Pipeline, so cross-validation and calibration re-fit them on training folds only. The universe is a subset of the held-out test set, so every score is out-of-sample (D-014).

## 3. Optimisation constraint checks (6.3)
`tests/test_validation.py` runs property tests on 35 random scenarios: random customers, costs, eligibility, budgets, capacities, segment maxima and minima, and protected-share caps. Every strategy's list passes the independent `check_constraints()`, and repair returns a feasible list from any starting chromosome. The default scenario's audit is below.

In [7]:
audit = pd.DataFrame({name: op.check_constraints(t, comp.selected_ids(name)).set_index('constraint')['ok']
                      for name in comp.selections})
print('All constraints met for every strategy:', bool(audit.all().all()))
audit.replace({True: '✓', False: '✗'})

All constraints met for every strategy: True


,Random,Propensity,Greedy,Rule-based,Genetic Algorithm,MILP optimum
constraint,,,,,,
Unique customers,✓,✓,✓,✓,✓,✓
Known customers,✓,✓,✓,✓,✓,✓
Eligible customers only,✓,✓,✓,✓,✓,✓
Budget (£),✓,✓,✓,✓,✓,✓
Contacts,✓,✓,✓,✓,✓,✓
Channel capacity: calls,✓,✓,✓,✓,✓,✓
Channel capacity: digital,✓,✓,✓,✓,✓,✓
Max share: High value,✓,✓,✓,✓,✓,✓
Max share: Growing,✓,✓,✓,✓,✓,✓


## 4. Backtest on actual outcomes (6.4)
The campaign universe holds real outcomes (`y`). For each strategy's list, compare the **expected** conversions and contribution (from P_i) with what these customers **actually** did.

Realised contribution = V_i summed over the customers who subscribed, minus the contact cost of everyone on the list.

*Caveat: `y` was recorded when these customers were contacted in the historical campaign. This tests whether P_i is accurate on the chosen lists. It is not a causal estimate of a new campaign's effect (D-015).*

In [8]:
bt = ev.backtest(t, comp)
bt[['contacts', 'expected_conversions', 'actual_conversions', 'expected_rate', 'actual_rate',
    'expected_contribution', 'realised_contribution', 'realised_vs_expected', 'realised_incremental_vs_random']].round(3)

,contacts,expected_conversions,actual_conversions,expected_rate,actual_rate,expected_contribution,realised_contribution,realised_vs_expected,realised_incremental_vs_random
strategy,,,,,,,,,
Random,5517.36,676.191,672.64,0.123,0.122,60659.833,59689.733,-0.016,0.000
Propensity,4275.00,838.592,845.00,0.196,0.198,91244.019,90856.450,-0.004,31166.717
Greedy,6000.00,884.167,883.00,0.147,0.147,103751.776,102877.200,-0.008,43187.467
Genetic Algorithm,6000.00,890.929,889.00,0.148,0.148,103793.081,102829.258,-0.009,43139.524
Rule-based,5374.00,756.167,780.00,0.141,0.145,83818.837,86567.130,0.033,26877.397
MILP optimum,6000.00,890.929,889.00,0.148,0.148,103793.081,102829.258,-0.009,43139.524


In [9]:
names = list(op.CHART_STRATEGIES)
fig = go.Figure()
fig.add_bar(x=names, y=bt.loc[names, 'expected_contribution'], name='Expected (from P_i)',
            marker=dict(color=ch.BLUE, cornerradius=4), hovertemplate='%{x}<br>expected £%{y:,.0f}<extra></extra>')
fig.add_bar(x=names, y=bt.loc[names, 'realised_contribution'], name='Realised (actual y)',
            marker=dict(color=ch.ORANGE, cornerradius=4), hovertemplate='%{x}<br>realised £%{y:,.0f}<extra></extra>')
ch.style(fig, 'Expected vs realised contribution by strategy', height=420)
fig.update_layout(barmode='group', bargap=0.3, bargroupgap=0.08)
ch.set_value_axis(fig, bt.loc[names, ['expected_contribution', 'realised_contribution']].max().max() * 1.12, money=True)
fig.show()

In [10]:
ci = ev.bootstrap_backtest(t, comp, ('Genetic Algorithm', 'Greedy', 'Propensity', 'Rule-based'))
rnd = bt.loc['Random']
ci['random_realised_mean'] = rnd['realised_contribution']
ci['random_realised_std'] = rnd['realised_contribution_std']
ci.round(0)

,realised,ci_low,ci_high,random_realised_mean,random_realised_std
strategy,,,,,
Genetic Algorithm,102829.0,93158.0,112433.0,59690.0,2370.0
Greedy,102877.0,93460.0,112261.0,59690.0,2370.0
Propensity,90856.0,82181.0,100526.0,59690.0,2370.0
Rule-based,86567.0,77118.0,96329.0,59690.0,2370.0


The backtest confirms the model's accuracy where it matters, on the selected customers: the GA list was expected to convert about 891 customers and 889 did. Its realised contribution is within about 1% of the expectation, and its 95% bootstrap interval lies well above Random's realised mean (about £60K, std about £2K across 50 seeds). The ranking also holds on real outcomes: GA ≈ Greedy > Propensity > Rule-based > Random. Greedy edges the GA on *realised* contribution by about £50, which is noise: the 95% intervals are about ±£10K.

## 5. Robustness (6.5)
### 5.1 GA across 5 seeds

In [11]:
runs = ev.ga_seed_runs(t, seeds=range(5), optimum=comp.milp.expected_profit)
summary = runs.groupby('seeded').agg(mean=('expected_contribution', 'mean'), std=('expected_contribution', 'std'),
                                     min_pct_of_optimum=('pct_of_optimum', 'min'),
                                     mean_generations=('generations', 'mean'), mean_runtime_s=('runtime_s', 'mean'),
                                     all_feasible=('feasible', 'all'))
summary.index = summary.index.map({True: 'Greedy-seeded', False: 'Unseeded'})
display(summary.round(4))
runs.round(4)

,mean,std,min_pct_of_optimum,mean_generations,mean_runtime_s,all_feasible
seeded,,,,,,
Unseeded,103793.0808,0.0,1.0,104.2,16.6224,True
Greedy-seeded,103793.0808,0.0,1.0,88.2,13.8238,True


,seeded,seed,expected_contribution,generations,runtime_s,feasible,pct_of_optimum
0,True,0,103793.0808,86,14.2461,True,1.0
1,True,1,103793.0808,87,13.1246,True,1.0
2,True,2,103793.0808,92,13.8257,True,1.0
3,True,3,103793.0808,86,13.5915,True,1.0
4,True,4,103793.0808,90,14.3310,True,1.0
5,False,0,103793.0808,104,16.4904,True,1.0
6,False,1,103793.0808,102,15.4130,True,1.0
7,False,2,103793.0808,107,18.0884,True,1.0
8,False,3,103793.0808,105,17.7067,True,1.0
9,False,4,103793.0808,103,15.4136,True,1.0


### 5.2 Does the strategy ranking survive different economics?
One driver changes at a time (±25%, ±50%), and everything else stays at the default. Segments are re-derived each time, because the value cut-off moves with the economics. Random is averaged over 20 seeds here, and the GA is capped at 30 s per case.

In [12]:
base = pl.load_universe().merge(pl.load_scored(), on='customer_id')
rs = ev.ranking_sensitivity(base)
rs[['change', 'parameter', 'ep_Random', 'ep_Propensity', 'ep_Greedy', 'ep_Genetic Algorithm', 'ep_Rule-based',
    'ga_rank', 'ga_vs_greedy', 'ga_vs_random_pct', 'all_feasible', 'ranking']].round(1)

,change,parameter,ep_Random,ep_Propensity,ep_Greedy,ep_Genetic Algorithm,ep_Rule-based,ga_rank,ga_vs_greedy,ga_vs_random_pct,all_feasible,ranking
0,default,default,60414.6,91244.0,103751.8,103793.1,83818.8,1,41.3,0.7,True,Genetic Algorithm > Greedy > Propensity > Rule...
1,-50%,spread,-9976.5,1757.7,18587.7,18594.8,-103.3,1,7.1,-2.9,True,Genetic Algorithm > Greedy > Propensity > Rule...
2,-25%,spread,25219.1,46500.9,57140.6,57140.6,41857.8,2,0.0,1.3,True,Greedy > Genetic Algorithm > Propensity > Rule...
3,+25%,spread,95610.2,135987.2,151227.6,152180.5,125779.9,1,952.9,0.6,True,Genetic Algorithm > Greedy > Propensity > Rule...
4,+50%,spread,130805.7,180730.3,198705.7,200716.2,167741.0,1,2010.5,0.5,True,Genetic Algorithm > Greedy > Propensity > Rule...
5,-50%,incentive,73911.3,108015.9,121438.8,121752.8,98942.2,1,314.0,0.6,True,Genetic Algorithm > Greedy > Propensity > Rule...
6,-25%,incentive,67163.0,99629.9,112594.1,112742.1,91380.5,1,148.0,0.7,True,Genetic Algorithm > Greedy > Propensity > Rule...
7,+25%,incentive,53666.3,82858.1,94912.4,94916.9,76257.2,1,4.6,0.8,True,Genetic Algorithm > Greedy > Propensity > Rule...
8,+50%,incentive,46917.9,74472.2,86076.3,86076.3,68695.5,2,0.0,0.8,True,Greedy > Genetic Algorithm > Propensity > Rule...
9,-50%,contact_cost,92627.9,128279.5,126933.8,129669.9,121909.8,1,2736.1,0.4,True,Genetic Algorithm > Propensity > Greedy > Rule...


In [13]:
print('GA first or tied first in', int(((rs.ga_rank == 1) | (rs.ga_vs_greedy >= -1e-6)).sum()), 'of', len(rs), 'cases')
print('GA never below Greedy:', bool((rs.ga_vs_greedy >= -1e-6).all()), '| every list feasible:', bool(rs.all_feasible.all()))
print('Propensity beats Greedy in:', rs.loc[rs['ep_Propensity'] > rs['ep_Greedy'], ['parameter', 'change']].to_dict('records') or 'none')
gain = rs['ep_Genetic Algorithm'] - rs['ep_Random']
print(f'GA gain over Random: £{gain.min():,.0f} to £{gain.max():,.0f}; largest GA gain over Greedy: £{rs.ga_vs_greedy.max():,.0f}')
print('Cases where Random or Rule-based lose money:', rs.loc[(rs['ep_Random'] < 0) | (rs['ep_Rule-based'] < 0), ['parameter', 'change']].to_dict('records') or 'none')

GA first or tied first in 13 of 13 cases
GA never below Greedy: True | every list feasible: True
Propensity beats Greedy in: [{'parameter': 'contact_cost', 'change': '-50%'}]
GA gain over Random: £28,571 to £69,910; largest GA gain over Greedy: £2,736
Cases where Random or Rule-based lose money: [{'parameter': 'spread', 'change': '-50%'}]


**The GA comes first, or tied first, in all 13 cases and is never below Greedy.** The rest of the ordering mostly holds (GA ≥ Greedy > Propensity > Rule-based > Random), with one exception: when contact costs halve, **Propensity beats Greedy**. With cheap contacts the binding limit is the 6,000-contact cap, not money, so ranking by profit per £ is the wrong rule and favours cheap contacts. The GA is not tied to one ranking rule, so it handles that case (+£2.7K over Greedy). This is the clearest example of why an optimiser is worth having over a fixed rule.

The size of the gain moves a lot. At half the funding spread, an untargeted (Random) or balance-first (Rule-based) campaign **loses money**, while the GA still earns about £18.6K. The funding spread is the assumption to confirm first.

## 6. Fairness monitor review (6.6)
Targeting rate by group, compared with the overall rate (about 66% of eligible customers are selected). A group is flagged if it is targeted at under half or over twice the overall rate, and only if it has at least 30 eligible customers. This is **reporting only**: no group label enters the optimiser, which only sees EP.

In [14]:
fm = ev.fairness_monitor(t, comp.selected_ids('Genetic Algorithm'))
print('Flagged groups:', fm[fm.flag != ''][['dimension', 'group', 'rate_ratio', 'flag']].to_dict('records') or 'none')
fm.round(3)

Flagged groups: none


,dimension,group,eligible,selected,share_of_eligible,share_of_selected,targeting_rate,rate_ratio,flag
0,age_band,18-29,976,874,0.108,0.146,0.895,1.349,
1,age_band,30-44,4862,3049,0.538,0.508,0.627,0.945,
2,age_band,45-59,2829,1704,0.313,0.284,0.602,0.908,
3,age_band,60-74,318,317,0.035,0.053,0.997,1.502,
4,age_band,75+,56,56,0.006,0.009,1.000,1.507,
5,job,admin,1023,715,0.113,0.119,0.699,1.053,
6,job,blue-collar,1850,1026,0.205,0.171,0.555,0.836,
7,job,entrepreneur,281,150,0.031,0.025,0.534,0.804,
8,job,housemaid,242,109,0.027,0.018,0.450,0.679,
9,job,management,2024,1540,0.224,0.257,0.761,1.147,


In [15]:
drv = pd.concat({'age_band': ev.fairness_drivers(t, 'age_band'), 'job': ev.fairness_drivers(t, 'job')})
rate = fm.set_index(['dimension', 'group'])['rate_ratio']
drv['rate_ratio'] = rate.reindex(drv.index).values
drv.round(3)

eligible  avg_propensity  avg_value  avg_expected_profit  pct_positive_ep  pct_digital  rate_ratio
age_band 18-29               976           0.195    147.916               20.380            0.921        0.285       1.349
         30-44              4862           0.105    154.398                7.759            0.711        0.285       0.945
         45-59              2829           0.095    160.795                7.172            0.697        0.291       0.908
         60-74               318           0.344    180.681               52.689            1.000        0.145       1.502
         75+                  56           0.497    177.894               78.686            1.000        0.000       1.507
job      admin              1023           0.128    149.629               10.638            0.773        0.290       1.053
         blue-collar        1850           0.077    151.158                3.907            0.635        0.390       0.836
         entrepreneur        281           0.081    157.140                4.381            0.637        0.317       0.804
         housemaid           242           0.083    155.523                4.342            0.558        0.310       0.679
         management         2024           0.143    163.874               14.621            0.826        0.208       1.147
         retired             468           0.248    174.936               36.047            0.893        0.231       1.298
         self-employed       341           0.118    161.513               10.457            0.713        0.270       0.937
         services            831           0.096    151.974                6.277            0.693        0.333       0.914
         student             184           0.295    154.367               35.921            0.995        0.168       1.499
         technician         1490           0.110    155.096                8.107            0.707        0.235       0.921
         unemployed          250           0.158    156.052               15.332            0.892        0.212       1.205
         unknown              57           0.143    165.330               15.869            0.895        0.333       1.242

In [16]:
fig = go.Figure()
for dim, color in (('age_band', ch.BLUE), ('job', ch.ORANGE)):
    f = fm[fm.dimension == dim]
    fig.add_bar(x=f['group'], y=f['rate_ratio'], name='Age band' if dim == 'age_band' else 'Job',
                marker=dict(color=color, cornerradius=3),
                customdata=f[['eligible', 'targeting_rate']],
                hovertemplate='%{x}<br>rate ratio %{y:.2f}<br>%{customdata[0]:,} eligible, %{customdata[1]:.0%} targeted<extra></extra>')
ch.reference_line(fig, -0.5, 1, 16.5, 1, 'overall rate', label_at='start')
for yv, lab in ((0.5, 'flag below 0.5×'), (2.0, 'flag above 2×')):
    ch.reference_line(fig, -0.5, yv, 16.5, yv, lab)
ch.style(fig, 'Targeting rate relative to the overall rate (GA list)', height=430)
fig.update_yaxes(title='Rate ratio', range=[0, 2.2])
fig.show()

**Review.** No group is flagged, and the skews follow the economics, not the group label:

- **Customers aged 60+, students and retired customers** are targeted at about 1.3-1.5×. Their propensity is 2-4× the average of about 12% (prior-campaign success is concentrated there). Every eligible 75+ customer has positive EP, so all 56 are selected. That is 0.9% of the list, under the 2% cap, and each one is flagged for human review before contact (D-007).
- **Housemaids, entrepreneurs and blue-collar workers** are targeted at about 0.7-0.85×. Their average propensity is about 8%, against 14-30% for management, retired customers and students.

**A limit of this monitor:** the list covers 66% of eligible customers, so no group can be targeted above 1/0.66 ≈ 1.51× the overall rate. The 2× over-targeting flag therefore cannot fire in the default scenario, and several groups are at that ceiling (every eligible member selected). The *share of list vs share of eligible* columns are the more informative view here. The 0.5× under-targeting flag still works. Housemaids, the lowest group at 0.68×, are not flagged.

A business owner should confirm these patterns are acceptable. If not, they can be capped with the same segment-share constraints the optimiser already supports.

## Summary
| Check | Result |
|---|---|
| Clean-clone reproduction | All headline outputs identical (metrics, EP, segments, every strategy, fingerprint) |
| Leakage | Chosen model ROC-AUC ≈ 0.75. With `duration` it would be ≈ 0.91: the leak we excluded |
| Transforms | Inside the Pipeline for both models; universe never seen in training |
| Constraints | Every strategy feasible in the default case and 35 random scenarios |
| Backtest | GA list: 891 expected vs 889 actual conversions; ranking holds on real outcomes |
| Robustness | GA reaches the optimum on all 10 seeded and unseeded runs. Under ±50% economics the GA is first or tied in 13/13 cases; Propensity overtakes Greedy when contacts are cheap |
| Fairness | No group flagged; skews explained by propensity; 75+ capped and flagged for review. The 2× flag can't fire when 66% of the eligible base is selected |